## Explanability
---

### Imports
---

In [ ]:
# system
import time
import os
import sys
ROOT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(ROOT_PATH)

#machine learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import shap

#scalers
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

#data manipulation
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
random_state = 42
sample_size = int(150e3)

### Load data
---

In [ ]:
risk_score_df = pd.read_csv(f"{ROOT_PATH}\\src\\data\\analysis\\customer_level_risk_score.csv")

In [ ]:
risk_score_df.head()

In [ ]:
risk_score_df.shape

### Training Forecast
---

In [ ]:
sample_df = risk_score_df #.sample(n=sample_size, random_state=random_state).reset_index(drop=True)

In [ ]:
rfr = RandomForestRegressor(
    n_estimators=60,
    max_depth=12,
    min_samples_leaf=10,
    max_features="sqrt",
    random_state=random_state,
    n_jobs=-1
)

In [ ]:
scaler = RobustScaler()
drop_columns = ["customer_id","hdbscan_cluster","hdbscan_risk_score","iso_risk_score","final_score"]
features_columns = [col for col in sample_df.columns if col not in drop_columns]
X = scaler.fit_transform(sample_df[features_columns])
y = sample_df["final_score"]

In [ ]:
rfr.fit(X, y)

In [ ]:
y_pred = rfr.predict(X)

In [ ]:
r2_score(y, y_pred)

### Adding SHAP
---

In [ ]:
threshold = 0.7
top_score_idx = sample_df[sample_df["final_score"] > threshold].index
print(f"Number of samples above threshold: {len(top_score_idx)}")

In [ ]:
top_risk_customer_df = sample_df.iloc[top_score_idx].reset_index(drop=True)

In [ ]:
shap_raw_x = top_risk_customer_df[features_columns].to_numpy()
shap_x = scaler.transform(shap_raw_x)

In [ ]:
print(f"SHAP version: {shap.__version__}")
print(f"Number of features: {shap_x.shape[1]}")
print(f"Number of samples: {shap_x.shape[0]}")

In [ ]:
explainer = shap.TreeExplainer(rfr)

In [ ]:
start = time.time()
shap_values = explainer.shap_values(shap_x)
print("Time:", time.time() - start)

In [ ]:
shap.summary_plot(shap_values, shap_x, plot_type='bar', feature_names=features_columns)

In [ ]:
shap.summary_plot(shap_values, shap_x, feature_names=features_columns)

In [ ]:
features_columns

In [ ]:
category_templates = {
    "transaction_intensity": 
        "The customer demonstrates elevated transaction intensity, including {}.",

    "transaction_size_pattern":
        "The customer shows atypical transaction size patterns, particularly {}.",

    "transaction_variability":
        "The customer exhibits high variability in transaction behavior, including {}.",

    "large_transactions":
        "Unusually large transactions were identified, particularly {}.",

    "fund_flow_pattern":
        "The customer presents an imbalanced fund flow structure, characterized by {}.",

    "short_term_activity":
        "Recent short-term activity shows unusual levels, including {}.",

    "activity_acceleration":
        "There is a significant acceleration in transaction behavior, evidenced by {}."
}

feature_metadata = {

# transaction volume

"transaction_count_received": {
    "theme": "transaction_intensity",
    "subtheme": "incoming_activity",
    "behavior": "number of incoming transactions",
    "risk_trigger": "high",
    "context_type": "peer_comparison",
    "description": "total number of received transactions"
},

"transaction_count_sent": {
    "theme": "transaction_intensity",
    "subtheme": "outgoing_activity",
    "behavior": "number of outgoing transactions",
    "risk_trigger": "high",
    "context_type": "peer_comparison",
    "description": "total number of sent transactions"
},

"total_amount_received": {
    "theme": "transaction_intensity",
    "subtheme": "incoming_volume",
    "behavior": "total incoming transaction volume",
    "risk_trigger": "high",
    "context_type": "peer_comparison",
    "description": "aggregate monetary value of received funds"
},

"total_amount_sent": {
    "theme": "transaction_intensity",
    "subtheme": "outgoing_volume",
    "behavior": "total outgoing transaction volume",
    "risk_trigger": "high",
    "context_type": "peer_comparison",
    "description": "aggregate monetary value of sent funds"
},

# transaction size behavior

"median_amount_received": {
    "theme": "transaction_size_pattern",
    "subtheme": "incoming_typical_size",
    "behavior": "typical incoming transaction size",
    "risk_trigger": "high",
    "context_type": "peer_comparison",
    "description": "median value of received transactions"
},

"median_amount_sent": {
    "theme": "transaction_size_pattern",
    "subtheme": "outgoing_typical_size",
    "behavior": "typical outgoing transaction size",
    "risk_trigger": "high",
    "context_type": "peer_comparison",
    "description": "median value of sent transactions"
},

"std_amount_received": {
    "theme": "transaction_variability",
    "subtheme": "incoming_variability",
    "behavior": "variability in incoming transaction sizes",
    "risk_trigger": "high",
    "context_type": "behavioral_deviation",
    "description": "standard deviation of received transaction values"
},

"std_amount_sent": {
    "theme": "transaction_variability",
    "subtheme": "outgoing_variability",
    "behavior": "variability in outgoing transaction sizes",
    "risk_trigger": "high",
    "context_type": "behavioral_deviation",
    "description": "standard deviation of sent transaction values"
},

"max_amount_received": {
    "theme": "transaction_variability",
    "subtheme": "incoming_large_value",
    "behavior": "largest incoming transaction amount",
    "risk_trigger": "high",
    "context_type": "peer_comparison",
    "description": "maximum received transaction value"
},

"max_amount_sent": {
    "theme": "transaction_variability",
    "subtheme": "outgoing_large_value",
    "behavior": "largest outgoing transaction amount",
    "risk_trigger": "high",
    "context_type": "peer_comparison",
    "description": "maximum sent transaction value"
},

# directional behavior

"sent_received_ratio": {
    "theme": "fund_flow_pattern",
    "subtheme": "directional_balance",
    "behavior": "ratio of outgoing to incoming transaction volume",
    "risk_trigger": "extreme",
    "context_type": "behavioral_deviation",
    "description": "measures imbalance between funds sent and received"
},

"transaction_direction_ratio": {
    "theme": "fund_flow_pattern",
    "subtheme": "transaction_direction_bias",
    "behavior": "ratio of outgoing to incoming transaction count",
    "risk_trigger": "extreme",
    "context_type": "behavioral_deviation",
    "description": "measures imbalance in number of sent vs received transactions"
},

# short term activity

"total_amount_received_30d": {
    "theme": "short_term_activity",
    "subtheme": "incoming_30d_volume",
    "behavior": "incoming transaction volume over last 30 days",
    "risk_trigger": "high",
    "context_type": "temporal_deviation",
    "description": "total received amount during the past 30 days"
},

"total_amount_received_7d": {
    "theme": "short_term_activity",
    "subtheme": "incoming_7d_volume",
    "behavior": "incoming transaction volume over last 7 days",
    "risk_trigger": "high",
    "context_type": "temporal_deviation",
    "description": "total received amount during the past 7 days"
},

"total_amount_sent_30d": {
    "theme": "short_term_activity",
    "subtheme": "outgoing_30d_volume",
    "behavior": "outgoing transaction volume over last 30 days",
    "risk_trigger": "high",
    "context_type": "temporal_deviation",
    "description": "total sent amount during the past 30 days"
},

"total_amount_sent_7d": {
    "theme": "short_term_activity",
    "subtheme": "outgoing_7d_volume",
    "behavior": "outgoing transaction volume over last 7 days",
    "risk_trigger": "high",
    "context_type": "temporal_deviation",
    "description": "total sent amount during the past 7 days"
},

"transaction_count_received_30d": {
    "theme": "short_term_activity",
    "subtheme": "incoming_30d_frequency",
    "behavior": "number of incoming transactions in last 30 days",
    "risk_trigger": "high",
    "context_type": "temporal_deviation",
    "description": "received transaction count in the past 30 days"
},

"transaction_count_received_7d": {
    "theme": "short_term_activity",
    "subtheme": "incoming_7d_frequency",
    "behavior": "number of incoming transactions in last 7 days",
    "risk_trigger": "high",
    "context_type": "temporal_deviation",
    "description": "received transaction count in the past 7 days"
},

"transaction_count_sent_30d": {
    "theme": "short_term_activity",
    "subtheme": "outgoing_30d_frequency",
    "behavior": "number of outgoing transactions in last 30 days",
    "risk_trigger": "high",
    "context_type": "temporal_deviation",
    "description": "sent transaction count in the past 30 days"
},

"transaction_count_sent_7d": {
    "theme": "short_term_activity",
    "subtheme": "outgoing_7d_frequency",
    "behavior": "number of outgoing transactions in last 7 days",
    "risk_trigger": "high",
    "context_type": "temporal_deviation",
    "description": "sent transaction count in the past 7 days"
},

# temporal acceleration

"total_amount_30d_ratio": {
    "theme": "activity_acceleration",
    "subtheme": "volume_change_30d",
    "behavior": "change in transaction volume compared to historical baseline (30 days)",
    "risk_trigger": "extreme",
    "context_type": "temporal_acceleration",
    "description": "measures recent increase or decrease in total transaction amount over 30 days"
},

"total_amount_7d_ratio": {
    "theme": "activity_acceleration",
    "subtheme": "volume_change_7d",
    "behavior": "change in transaction volume compared to historical baseline (7 days)",
    "risk_trigger": "extreme",
    "context_type": "temporal_acceleration",
    "description": "measures recent increase or decrease in total transaction amount over 7 days"
},

"transaction_count_30d_ratio": {
    "theme": "activity_acceleration",
    "subtheme": "frequency_change_30d",
    "behavior": "change in transaction frequency compared to historical baseline (30 days)",
    "risk_trigger": "extreme",
    "context_type": "temporal_acceleration",
    "description": "measures recent increase or decrease in transaction count over 30 days"
},

"transaction_count_7d_ratio": {
    "theme": "activity_acceleration",
    "subtheme": "frequency_change_7d",
    "behavior": "change in transaction frequency compared to historical baseline (7 days)",
    "risk_trigger": "extreme",
    "context_type": "temporal_acceleration",
    "description": "measures recent increase or decrease in transaction count over 7 days"
}
}

In [ ]:
def get_top_contributing_features(shap_values, feature_columns, top_n=3):
    shap_df = pd.DataFrame({"feature": feature_columns, "shap_value": shap_values})
    top_features = shap_df.sort_values(by="shap_value", ascending=False).head(top_n)["feature"].tolist()
    top_features.sort()
    return top_features

def generate_text(top_features):
    explanations = {}
    for feature in top_features:
        metadata = feature_metadata.get(feature)
        if metadata:
            category = metadata.get("theme", "general")
            if category not in explanations:
                explanations[category] = []             
            explanations[category].append(metadata["description"])
    return explanations

def generate_explanation(shap_values, feature_columns, top_n=3):
    top_features = get_top_contributing_features(shap_values, feature_columns, top_n)
    explanation_dict = generate_text(top_features)
    paragraphs = []
    for category, descriptions in explanation_dict.items():
        template = category_templates.get(category)
        if template: 
            joined_desc = ", ".join(descriptions)
            paragraphs.append(template.format(joined_desc))
    return " ".join(paragraphs)

In [ ]:
top_risk_customer_df["top_features"] = [
    get_top_contributing_features(
        shap_values=shap_values[i],
        feature_columns=features_columns
    )
    for i in range(len(top_risk_customer_df))
]

top_risk_customer_df["explanation"] = [
    generate_explanation(
        shap_values=shap_values[i],
        feature_columns=features_columns
    )
    for i in range(len(top_risk_customer_df))
]

In [ ]:
top_risk_customer_df.to_csv("./test.csv", index=False)